In [ ]:
# ------------------------------------------------------------------
# [셀 1] 필수 패키지 설치 (Colab 또는 신규 노트북 환경일 경우 실행)
# ------------------------------------------------------------------
!pip install ultralytics opencv-python matplotlib

In [ ]:
# ------------------------------------------------------------------
# [셀 1] Roboflow 라이브러리 설치 및 데이터셋 다운로드 (예시)
# - Roboflow 웹사이트의 [Download Dataset] -> [YOLOv11]에서 제공하는
#   내 프로젝트 전용 코드를 가져와 복사해 넣습니다.
# ------------------------------------------------------------------

# Roboflow 파이썬 SDK 패키지 설치
!pip install roboflow
"""
from roboflow import Roboflow
# 1. API 키 설정 (예시: 본인의 Roboflow API Key 입력)
rf = Roboflow(api_key="1OfJanF0XJZmfOOP7Crw")
# 2. 워크스페이스 및 프로젝트 지정 (예시: Roboflow 내 프로젝트 이름)
project = rf.workspace("first-kefyw").project("-seg-2z7dd-aw6dh-fork")
# 3. 데이터셋 버전에 접근 (예시: 버전 번호)
version = project.version(1)
# 4. YOLOv11 포맷으로 데이터셋 자동 다운로드 (현재 작업 디렉토리에 폴더 생성)
dataset = version.download("yolov11")
"""

In [ ]:
import os
import json
import cv2
from ultralytics import YOLO


class YOLOSegmentPipeline:

    def __init__(self, model_path: str = "yolo11m-seg.pt", base_dir: str = "runs", exp_name: str = "yolo11_climbing_seg"):
        self.model_path = model_path
        self.model = YOLO(model_path)
        
        self.base_dir = base_dir
        self.exp_name = exp_name
        
        # os.path.join(): OS(Linux/Windows) 환경에 맞춰 경로 구분자(/ 또는 \)를 자동 결합
        self.exp_dir = os.path.join(self.base_dir, self.exp_name)
        
        self.train_dir = os.path.join(self.exp_dir, "train")
        self.val_dir = os.path.join(self.exp_dir, "val")
        self.predict_dir = os.path.join(self.exp_dir, "predict")

    def train(self, data_yaml: str
              , epochs: int = 100
              , imgsz: int = 640
              , batch: int = 16
              , device: str | int = 0
              , patience: int = 20
              , resume: bool = False) : 
        print(f"\n--- [Train] 모델 학습 시작 (출력 경로: {self.train_dir}) ---")
        
        results = self.model.train(
              data=data_yaml
            , epochs=epochs
            , imgsz=imgsz
            , batch=batch
            , project=self.exp_dir
            , name="train"
            , exist_ok=True
            , device=device
            , patience=patience
            , save=True
            , resume=resume
        )

        # train을 지나면 best.pt로 모델을 변경한다
        best_weights = os.path.join(self.train_dir, "weights", "best.pt")
        if os.path.exists(best_weights):
            self.model_path = best_weights
            self.model = YOLO(best_weights)

        return results

    def validate(self, data_yaml: str, split: str = "val", device: str | int = 0) -> dict:
        print(f"\n--- [Val] 모델 성능 검증 시작 (출력 경로: {self.val_dir}) ---")
        
        # os.makedirs(): 디렉토리 생성 (exist_ok=True: 이미 폴더가 존재해도 예외 없이 무시)
        os.makedirs(self.val_dir, exist_ok=True)
        
        metrics = self.model.val(
            data=data_yaml, split=split, project=self.exp_dir,
            name="val", exist_ok=True, device=device
        )

        performance_summary = {
            "box_map50_95": float(metrics.box.map),
            "box_map50": float(metrics.box.map50),
            "seg_map50_95": float(metrics.seg.map),
            "seg_map50": float(metrics.seg.map50),
            "precision": float(metrics.box.p.mean()) if hasattr(metrics.box.p, 'mean') else 0.0,
            "recall": float(metrics.box.r.mean()) if hasattr(metrics.box.r, 'mean') else 0.0
        }

        output_json = os.path.join(self.val_dir, "val_metrics.json")
        with open(output_json, "w", encoding="utf-8") as f:
            # json.dump(): 파이썬 딕셔너리 데이터를 JSON 파일로 직렬화 (indent=4: 들여쓰기 4칸 적용)
            json.dump(performance_summary, f, ensure_ascii=False, indent=4)

        return performance_summary

    def predict_and_crop(self, source: str | list, conf: float = 0.5, imgsz: int = 640, device: str | int = 0) -> list:
        print(f"\n--- [Predict] 추론 및 인스턴스 크롭 시작 (출력 경로: {self.predict_dir}) ---")
        
        crop_dir = os.path.join(self.predict_dir, "crops")
        # os.makedirs(): 크롭 이미지를 저장할 폴더가 없으면 자동 생성
        os.makedirs(crop_dir, exist_ok=True)

        results = self.model.predict(
            source=source, conf=conf, imgsz=imgsz, project=self.exp_dir,
            name="predict", exist_ok=True, save=True, save_txt=False, device=device
        )

        all_images_data = []

        for r_idx, result in enumerate(results):
            boxes = result.boxes
            masks = result.masks
            image_path = result.path
            
            # cv2.imread(): OpenCV를 이용해 이미지를 NumPy 배열 형태로 디스크에서 로드
            orig_img = cv2.imread(image_path)
            if orig_img is None:
                continue

            h, w, _ = orig_img.shape
            img_result = {
                "image_index": r_idx,
                "image_path": image_path,
                "detected_objects": []
            }

            if boxes is not None and len(boxes) > 0:
                for i in range(len(boxes)):
                    class_id = int(boxes.cls[i].item())
                    class_name = self.model.names[class_id]
                    confidence = float(boxes.conf[i].item())
                    xyxy = boxes.xyxy[i].tolist()

                    polygon = None
                    if masks is not None and len(masks.xy) > i:
                        polygon = masks.xy[i].tolist()

                    cropped_filename = None
                    x1, y1, x2, y2 = map(int, xyxy)
                    x1, y1 = max(0, x1), max(0, y1)
                    x2, y2 = min(w, x2), min(h, y2)

                    if x2 > x1 and y2 > y1:
                        cropped_box = orig_img[y1:y2, x1:x2]
                        
                        # os.path.splitext(): 경로에서 '파일명'과 '.확장자'를 분리 (예: 'sample.jpg' -> ('sample', '.jpg'))
                        base_name = os.path.splitext(os.path.basename(image_path))[0]
                        
                        cropped_filename = os.path.join(
                            crop_dir, f"{base_name}_obj{i}_{class_name}.jpg"
                        )
                        # cv2.imwrite(): NumPy 배열 형태의 자른 이미지 데이터를 파일로 저장
                        cv2.imwrite(cropped_filename, cropped_box)

                    obj_info = {
                        "class_id": class_id,
                        "class_name": class_name,
                        "confidence": round(confidence, 4),
                        "bounding_box": [round(c, 2) for c in xyxy],
                        "polygon": polygon,
                        "cropped_image_path": cropped_filename
                    }
                    img_result["detected_objects"].append(obj_info)

            all_images_data.append(img_result)

        output_json = os.path.join(self.predict_dir, "inference_results.json")
        with open(output_json, "w", encoding="utf-8") as f:
            # json.dump(): 추론 결과 메타데이터를 JSON 형식으로 저장
            json.dump(all_images_data, f, ensure_ascii=False, indent=4)

        return all_images_data


# ==========================================
# 스크립트 직접 실행부 (Main)   /    train,val,test를 동시에 진행한다면 best.pt가 만들어 지고 그것으로 모델을 변경한 뒤에 val가 test를 진행함
# ==========================================
if __name__ == "__main__":
    # 데이터셋 및 추론 대상 경로 설정
    yaml_path = "/workspace/hold_seg_data-1/data.yaml"
    test_source = "/workspace/hold_seg_data-1/test/images"

    # [파이프라인 인스턴스 초기화]
    pipeline = YOLOSegmentPipeline(
        model_path="yolo11m-seg.pt",        # 베이스 모델 또는 학습된 가중치(.pt) 경로
        base_dir="runs",                    # 최상위 결과 저장 디렉토리
        exp_name="yolo11_climbing_seg"       # 실험 프로젝트 명칭
    )

    # ------------------------------------------
    # 1. 모델 학습 (Train)
    # ------------------------------------------
    '''pipeline.train(
        data_yaml=yaml_path,  # dataset 설정 파일 경로
        epochs=100,           # 총 학습 에폭 수
        imgsz=640,            # 입력 이미지 리사이즈 크기
        batch=16,             # 배치 사이즈
        device=0,             # GPU 장치 번호 (또는 'cpu')
        patience=20,          # 조기 종료(Early Stopping) 대기 에폭
        resume=False          # last.pt부터 이어서 학습할지 여부
    )'''

    # ------------------------------------------
    # 2. 성능 검증 (Val)
    # ------------------------------------------
    pipeline.validate(
        data_yaml=yaml_path,    # dataset 설정 파일 경로
        split="val",            # 검증 데이터셋 분할 영역 ('val' 또는 'test')
        device=0                # GPU 장치 번호 (또는 'cpu')
    )

    # ------------------------------------------
    # 3. 추론 및 인스턴스 크롭 (Predict & Crop)
    # ------------------------------------------
    pipeline.predict_and_crop(
        source=test_source,     # 추론 대상 (단일 이미지 경로, 이미지 리스트, 폴더 경로)
        conf=0.5,               # 탐지 신뢰도 임계값 (Confidence Threshold)
        imgsz=640,              # 추론시 입력 이미지 리사이즈 크기
        device=0                # GPU 장치 번호 (또는 'cpu')
    )